# Pila monótona

Una pila monótona guarda índices cuyos valores están siempre ordenados (crecientes o decrecientes). Al llegar un elemento nuevo se desapilan los que rompen el orden, y en ese momento cada desapilado descubre su "siguiente mayor" o "siguiente menor". Cada índice entra y sale una sola vez, así que todo el recorrido cuesta $O(n)$ en lugar del $O(n^2)$ de buscar hacia la derecha desde cada posición. Con esa idea se resuelven el siguiente elemento mayor, el mayor rectángulo de un histograma o la suma de los mínimos de todos los subarrays.

## 1. Los cuatro vecinos más cercanos

Escribir una función que, para cada posición, devuelva el índice del siguiente elemento estrictamente mayor, siguiente menor, anterior mayor y anterior menor (o `-1` si no existe), con una sola función parametrizada por el sentido y la comparación. Comprobarla contra la búsqueda directa.

In [1]:
import operator
import random


def vecino(a, compara, hacia_la_derecha=True):
    """Índice del vecino más cercano j con compara(a[j], a[i]) en el sentido indicado."""
    n = len(a)
    resultado = [-1] * n
    pila = []
    orden = range(n) if hacia_la_derecha else range(n - 1, -1, -1)
    for i in orden:
        while pila and compara(a[i], a[pila[-1]]):
            resultado[pila.pop()] = i
        pila.append(i)
    return resultado


def vecino_bruto(a, compara, hacia_la_derecha=True):
    n = len(a)
    resultado = []
    for i in range(n):
        candidatos = range(i + 1, n) if hacia_la_derecha else range(i - 1, -1, -1)
        resultado.append(next((j for j in candidatos if compara(a[j], a[i])), -1))
    return resultado


casos = {
    "siguiente mayor": (operator.gt, True),
    "siguiente menor": (operator.lt, True),
    "anterior mayor": (operator.gt, False),
    "anterior menor": (operator.lt, False),
}
alturas = [4, 2, 7, 7, 1, 5, 3, 8]
print(f"{'':16}", alturas)
for nombre, (compara, derecha) in casos.items():
    print(f"{nombre:16}", vecino(alturas, compara, derecha))

random.seed(1)
for _ in range(2000):
    a = [random.randint(0, 6) for _ in range(random.randint(0, 12))]
    for compara, derecha in casos.values():
        assert vecino(a, compara, derecha) == vecino_bruto(a, compara, derecha)

                 [4, 2, 7, 7, 1, 5, 3, 8]
siguiente mayor  [2, 2, 7, 7, 5, 7, 7, -1]
siguiente menor  [1, 4, 4, 4, -1, 6, -1, -1]
anterior mayor   [-1, 0, -1, -1, 3, 3, 5, -1]
anterior menor   [-1, -1, 1, 1, -1, 4, 4, 6]


## 2. Cada índice entra y sale una vez

Contar las operaciones de apilar y desapilar al calcular el siguiente mayor en listas de distintos tamaños y comprobar que nunca pasan de $2n$, aunque haya bucles anidados en el código. Compararlo con las comparaciones de la fuerza bruta en el peor caso (una lista decreciente).

In [2]:
def operaciones_pila(a):
    pila, operaciones = [], 0
    for i, x in enumerate(a):
        while pila and x > a[pila[-1]]:
            pila.pop()
            operaciones += 1
        pila.append(i)
        operaciones += 1
    return operaciones


print(f"{'n':>6} {'pila':>8} {'bruta (decreciente)':>20}")
for n in (10, 100, 1000, 10_000):
    a = [random.randint(0, 100) for _ in range(n)]
    decreciente = list(range(n, 0, -1))
    ops = max(operaciones_pila(a), operaciones_pila(decreciente))
    bruta = n * (n - 1) // 2
    print(f"{n:>6} {ops:>8} {bruta:>20}")
    assert ops <= 2 * n

     n     pila  bruta (decreciente)
    10       16                   45
   100      194                 4950
  1000     1986               499500
 10000    19905             49995000


## 3. Siguiente mayor en una lista circular

Las temperaturas máximas de una semana en La Laguna se repiten semana tras semana. Para cada día, buscar la siguiente temperatura más alta dando la vuelta a la lista si hace falta: recorrerla dos veces con índices módulo $n$.

In [3]:
def siguiente_mayor_circular(a):
    n = len(a)
    resultado = [None] * n
    pila = []
    for k in range(2 * n):
        i = k % n
        while pila and a[i] > a[pila[-1]]:
            resultado[pila.pop()] = a[i]
        if k < n:
            pila.append(i)
    return resultado


def siguiente_mayor_circular_bruto(a):
    n = len(a)
    return [next((a[(i + d) % n] for d in range(1, n) if a[(i + d) % n] > a[i]), None) for i in range(n)]


dias = ["lun", "mar", "mié", "jue", "vie", "sáb", "dom"]
semana = [21, 23, 19, 25, 22, 25, 20]
for dia, t, s in zip(dias, semana, siguiente_mayor_circular(semana)):
    print(f"{dia}: {t} °C → {s if s is not None else 'ninguna'}")

for _ in range(2000):
    a = [random.randint(0, 5) for _ in range(random.randint(0, 10))]
    assert siguiente_mayor_circular(a) == siguiente_mayor_circular_bruto(a)

lun: 21 °C → 23
mar: 23 °C → 25
mié: 19 °C → 25
jue: 25 °C → ninguna
vie: 22 °C → 25
sáb: 25 °C → ninguna
dom: 20 °C → 21


## 4. Días hasta un día más cálido

Dadas las temperaturas diarias de un mes en Santa Cruz, calcular para cada día cuántos días hay que esperar hasta uno más cálido (0 si no llega). Es el siguiente mayor expresado como distancia.

In [4]:
def dias_hasta_mas_calido(t):
    espera = [0] * len(t)
    pila = []
    for i, x in enumerate(t):
        while pila and x > t[pila[-1]]:
            j = pila.pop()
            espera[j] = i - j
        pila.append(i)
    return espera


random.seed(7)
mes = [random.randint(19, 27) for _ in range(14)]
espera = dias_hasta_mas_calido(mes)
print("día     ", " ".join(f"{d:>2}" for d in range(1, 15)))
print("°C      ", " ".join(f"{x:>2}" for x in mes))
print("esperar ", " ".join(f"{x:>2}" for x in espera))

for _ in range(2000):
    t = [random.randint(15, 20) for _ in range(random.randint(0, 15))]
    bruto = [next((j - i for j in range(i + 1, len(t)) if t[j] > t[i]), 0) for i in range(len(t))]
    assert dias_hasta_mas_calido(t) == bruto

día       1  2  3  4  5  6  7  8  9 10 11 12 13 14
°C       24 21 25 19 20 27 20 24 19 27 22 19 20 25
esperar   2  1  3  1  1  0  1  2  1  0  3  1  1  0


## 5. Racha de precios en línea

Crear una clase `Racha` que reciba el precio del kilo de plátano día a día y devuelva, al llegar cada precio, cuántos días consecutivos (incluido el de hoy) lleva el precio siendo menor o igual que el actual. La pila guarda pares (precio, racha) y absorbe las rachas de los precios que supera.

In [5]:
class Racha:
    def __init__(self):
        self.pila = []

    def siguiente(self, precio):
        racha = 1
        while self.pila and self.pila[-1][0] <= precio:
            racha += self.pila.pop()[1]
        self.pila.append((precio, racha))
        return racha


precios = [0.95, 0.90, 1.05, 1.00, 1.00, 1.10, 0.85, 1.20]
r = Racha()
print([r.siguiente(p) for p in precios])

for _ in range(1000):
    p = [random.randint(1, 6) for _ in range(random.randint(1, 15))]
    r = Racha()
    rachas = [r.siguiente(x) for x in p]
    bruto = []
    for i in range(len(p)):
        k = i
        while k >= 0 and p[k] <= p[i]:
            k -= 1
        bruto.append(i - k)
    assert rachas == bruto

[1, 1, 3, 1, 2, 6, 1, 8]


## 6. Mayor rectángulo en un histograma

Las barras de un histograma tienen ancho 1 y las alturas dadas. Encontrar el rectángulo de mayor área que cabe dentro: para cada barra, el rectángulo que la usa como la más baja se extiende hasta el anterior menor y el siguiente menor. Con una pila creciente se obtienen ambos límites en una pasada. Comprobarlo contra probar todos los tramos $[i, j]$.

In [6]:
def mayor_rectangulo(alturas):
    """Devuelve (área, inicio, fin) con el tramo [inicio, fin)."""
    mejor = (0, 0, 0)
    pila = []  # índices con alturas crecientes
    for i, h in enumerate([*alturas, 0]):
        while pila and alturas[pila[-1]] >= h:
            j = pila.pop()
            izquierda = pila[-1] + 1 if pila else 0
            mejor = max(mejor, (alturas[j] * (i - izquierda), izquierda, i))
        pila.append(i)
    return mejor


def mayor_rectangulo_bruto(alturas):
    mejor = 0
    for i in range(len(alturas)):
        minimo = float("inf")
        for j in range(i, len(alturas)):
            minimo = min(minimo, alturas[j])
            mejor = max(mejor, minimo * (j - i + 1))
    return mejor


barras = [2, 1, 5, 6, 2, 3, 4, 1]
area, i, j = mayor_rectangulo(barras)
print(f"Área {area} en las barras {i}..{j - 1}, altura {min(barras[i:j])}")
for fila in range(max(barras), 0, -1):
    print("".join("█" if h >= fila and i <= k < j and fila <= min(barras[i:j]) else ("▒" if h >= fila else " ") for k, h in enumerate(barras)))

for _ in range(2000):
    a = [random.randint(0, 8) for _ in range(random.randint(0, 12))]
    area, i, j = mayor_rectangulo(a)
    assert area == mayor_rectangulo_bruto(a)
    assert area == 0 or area == min(a[i:j]) * (j - i)

Área 10 en las barras 2..6, altura 2
   ▒    
  ▒▒    
  ▒▒  ▒ 
  ▒▒ ▒▒ 
▒ █████ 
▒▒█████▒


## 7. Parcela libre más grande en un invernadero

Un invernadero se representa como una matriz de `1` (suelo libre) y `0` (ocupado). Encontrar el rectángulo de unos de mayor área: para cada fila, la altura de unos acumulados hacia arriba en cada columna forma un histograma, y se aplica el ejercicio anterior. Cuesta $O(filas \cdot columnas)$. Comprobarlo contra la fuerza bruta en matrices aleatorias.

In [7]:
def mayor_rectangulo_unos(matriz):
    if not matriz:
        return 0
    alturas = [0] * len(matriz[0])
    mejor = 0
    for fila in matriz:
        alturas = [h + 1 if celda else 0 for h, celda in zip(alturas, fila)]
        mejor = max(mejor, mayor_rectangulo(alturas)[0])
    return mejor


def mayor_rectangulo_unos_bruto(m):
    filas, cols = len(m), len(m[0])
    mejor = 0
    for f1 in range(filas):
        for c1 in range(cols):
            for f2 in range(f1, filas):
                for c2 in range(c1, cols):
                    if all(m[f][c] for f in range(f1, f2 + 1) for c in range(c1, c2 + 1)):
                        mejor = max(mejor, (f2 - f1 + 1) * (c2 - c1 + 1))
    return mejor


invernadero = [
    [1, 0, 1, 1, 1, 0],
    [1, 1, 1, 1, 1, 1],
    [0, 1, 1, 1, 1, 0],
    [1, 1, 1, 0, 1, 1],
]
for fila in invernadero:
    print(" ".join("·" if c else "#" for c in fila))
print("Área libre máxima:", mayor_rectangulo_unos(invernadero))
assert mayor_rectangulo_unos(invernadero) == 9

for _ in range(300):
    filas, cols = random.randint(1, 5), random.randint(1, 5)
    m = [[int(random.random() < 0.7) for _ in range(cols)] for _ in range(filas)]
    assert mayor_rectangulo_unos(m) == mayor_rectangulo_unos_bruto(m)

· # · · · #
· · · · · ·
# · · · · #
· · · # · ·
Área libre máxima: 9


## 8. Suma de mínimos de todos los subarrays

Calcular la suma de $\min(a_i, \dots, a_j)$ para todos los subarrays contiguos. Cada $a_k$ es el mínimo de $(k - p)(s - k)$ subarrays, siendo $p$ el anterior menor y $s$ el siguiente menor. Con valores repetidos hay que romper el empate de forma asimétrica (estricto a un lado y no estricto al otro) para no contar dos veces el mismo subarray. Comprobarlo contra la fuerza bruta y mostrar qué pasa si ambos lados son estrictos.

In [8]:
def suma_minimos(a, asimetrico=True):
    n = len(a)
    # anterior menor estricto; siguiente menor o igual (o estricto si no se rompe el empate)
    anterior = vecino(a, operator.lt, hacia_la_derecha=False)
    siguiente = [n] * n
    pila = []
    for i, x in enumerate(a):
        while pila and (x <= a[pila[-1]] if asimetrico else x < a[pila[-1]]):
            siguiente[pila.pop()] = i
        pila.append(i)
    return sum(a[k] * (k - anterior[k]) * (siguiente[k] - k) for k in range(n))


def suma_minimos_bruta(a):
    return sum(min(a[i:j + 1]) for i in range(len(a)) for j in range(i, len(a)))


print(suma_minimos([3, 1, 2, 4]), suma_minimos_bruta([3, 1, 2, 4]))
print("Con repetidos [2, 2, 2]:", suma_minimos([2, 2, 2]), "bien;", suma_minimos([2, 2, 2], asimetrico=False), "si ambos lados son estrictos")

for _ in range(2000):
    a = [random.randint(0, 5) for _ in range(random.randint(0, 12))]
    assert suma_minimos(a) == suma_minimos_bruta(a)

17 17
Con repetidos [2, 2, 2]: 12 bien; 20 si ambos lados son estrictos


## 9. Suma de amplitudes de todos los subarrays

La amplitud de un subarray es su máximo menos su mínimo. Sumar las amplitudes de todos los subarrays como "suma de máximos menos suma de mínimos", obteniendo la suma de máximos con la misma función aplicada a los valores negados. Comprobarlo contra la fuerza bruta con las mareas de una semana en el puerto de Arrecife (en centímetros).

In [9]:
def suma_amplitudes(a):
    return -suma_minimos([-x for x in a]) - suma_minimos(a)


mareas = [120, 85, 210, 60, 190, 95, 230]
print("Suma de amplitudes:", suma_amplitudes(mareas))
assert suma_amplitudes(mareas) == sum(max(mareas[i:j]) - min(mareas[i:j]) for i in range(7) for j in range(i + 1, 8))

for _ in range(1000):
    a = [random.randint(-5, 5) for _ in range(random.randint(0, 10))]
    assert suma_amplitudes(a) == sum(max(a[i:j]) - min(a[i:j]) for i in range(len(a)) for j in range(i + 1, len(a) + 1))

Suma de amplitudes: 2940


## 10. Agua atrapada entre muros

Unos muros de alturas dadas forman un perfil en el que se acumula agua de lluvia. Calcular el agua atrapada con una pila decreciente: al llegar un muro más alto que la cima, el hueco que queda entre el muro desapilado, el nuevo y el anterior de la pila se llena por capas horizontales. Comprobarlo contra la fórmula directa $\sum_i \min(\max_{izq}, \max_{der}) - h_i$.

In [10]:
def agua_atrapada(h):
    agua = 0
    pila = []
    for i, x in enumerate(h):
        while pila and x > h[pila[-1]]:
            fondo = pila.pop()
            if not pila:
                break
            izquierda = pila[-1]
            agua += (min(h[izquierda], x) - h[fondo]) * (i - izquierda - 1)
        pila.append(i)
    return agua


def agua_atrapada_directa(h):
    return sum(max(0, min(max(h[:i + 1]), max(h[i:])) - h[i]) for i in range(len(h)))


muros = [0, 3, 0, 1, 0, 2, 4, 1, 1, 3, 0, 2]
print("Agua:", agua_atrapada(muros))
nivel = [min(max(muros[:i + 1]), max(muros[i:])) for i in range(len(muros))]
for fila in range(max(muros), 0, -1):
    print("".join("█" if h >= fila else ("~" if n >= fila else " ") for h, n in zip(muros, nivel)))

for _ in range(2000):
    h = [random.randint(0, 6) for _ in range(random.randint(0, 12))]
    assert agua_atrapada(h) == agua_atrapada_directa(h)

Agua: 15
      █     
 █~~~~█~~█  
 █~~~██~~█~█
 █~█~█████~█


## 11. Quitar k dígitos para obtener el menor número

Dado un número como cadena, quitar exactamente $k$ dígitos para que lo que queda sea lo más pequeño posible. Con una pila creciente, cada dígito expulsa a los anteriores mayores mientras queden eliminaciones. Comprobarlo contra probar todas las combinaciones de posiciones.

In [11]:
from itertools import combinations


def quitar_digitos(numero, k):
    pila = []
    for d in numero:
        while k and pila and pila[-1] > d:
            pila.pop()
            k -= 1
        pila.append(d)
    resultado = "".join(pila[:len(pila) - k]).lstrip("0")
    return resultado or "0"


def quitar_digitos_bruto(numero, k):
    return str(min(int("".join(numero[i] for i in pos) or "0") for pos in combinations(range(len(numero)), len(numero) - k)))


for numero, k in [("1432219", 3), ("10200", 1), ("10", 2), ("38001", 2)]:
    print(f"{numero} sin {k} dígitos → {quitar_digitos(numero, k)}")

for _ in range(1000):
    numero = "".join(random.choice("0123456789") for _ in range(random.randint(1, 8)))
    k = random.randint(0, len(numero))
    assert quitar_digitos(numero, k) == quitar_digitos_bruto(numero, k)

1432219 sin 3 dígitos → 1219
10200 sin 1 dígitos → 200
10 sin 2 dígitos → 0
38001 sin 2 dígitos → 1
